<a href="https://colab.research.google.com/github/sarahibdah/Flyrank-ML-Internship-Sarah/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sarahibdah/Flyrank-ML-Internship-Sarah/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
import os, json, warnings
warnings.filterwarnings("ignore")
import pandas as pd, numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import precision_score, recall_score

url = "https://raw.githubusercontent.com/sarahibdah/Flyrank-ML-Internship-Sarah/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)
down = (df["trend_direction"] == "down")            # measured decline: last 30 days vs the 30 days before
y = down.astype(int)
base_rate = y.mean()
print(f"{len(df):,} pages, {df['client_id'].nunique()} clients, share declining = {base_rate:.3f}")

# Impressions lost = how far the last 30 days fell below the previous 30 days (observed, not a forecast).
df["lost_impressions"] = (df["impressions_prev_30d"] - df["impressions_last_30d"]).clip(lower=0)

# Secondary signal: tree built ONLY from windows before the last 30 days (ML-09's legal features),
# scored out-of-fold so every page is scored by a model that never saw its client.
df["imp_61_90"] = df["impressions_90d"] - df["impressions_last_30d"] - df["impressions_prev_30d"]
legal = ["age_tier_order", "impressions_prev_30d", "imp_61_90"]
df["p_tree"] = 0.0
for tr, te in GroupKFold(5).split(df, y, df["client_id"]):
    m = DecisionTreeClassifier(max_depth=4, min_samples_leaf=50, random_state=42).fit(df.iloc[tr][legal], y.iloc[tr])
    df.loc[df.index[te], "p_tree"] = m.predict_proba(df.iloc[te][legal])[:, 1]
tree_pred = (df["p_tree"] >= 0.5).astype(int)
print(f"Tree (legal features, grouped out-of-fold): precision {precision_score(y, tree_pred):.3f}, "
      f"recall {recall_score(y, tree_pred):.3f}, base rate {base_rate:.3f}")


30,000 pages, 32 clients, share declining = 0.542
Tree (legal features, grouped out-of-fold): precision 0.645, recall 0.861, base rate 0.542


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

**How the queue works.** Each page gets **one reason code**, one **action**, and a **priority**. The action comes from a rule on *measured* decline (the last 30 days against the 30 days before). It is triage of what already happened, not a forecast. The ML-09 tree is a secondary column, `model_support`, built only from earlier windows, so a reviewer can see whether the older history agrees.

| Reason code | When it applies | Action |
|---|---|---|
| `decline_large_loss` | declining and lost at least 500 impressions vs the previous 30 days | `review_for_refresh` |
| `decline_moderate_loss` | declining and lost 100 to 499 impressions | `watch` |
| `decline_small_loss` | declining but lost under 100 impressions (too small to read as a trend) | `no_action` |
| `not_declining` | stable, up, new or flat | `no_action` |

The 100 and 500 cut-offs are first guesses I chose, not tuned values. **Priority** is impressions lost, largest first, with the tree probability breaking ties. I chose this over Week 4's score because that score ranked by raw traffic, so the biggest pages came first regardless of how much traffic each had lost.

**Archetype to action (life-stage groups).** My lane has no clustering step, so my archetypes are life-stage groups from age tier. The "check first" column holds starting hypotheses for the reviewer. I have **not** tested them.

| Life stage | Age tier | Check first |
|---|---|---|
| young | 31-180 days | does the page match search intent and is it indexed? A fresh page may not need a rewrite |
| mid | 181-365 days | are facts, dates and links out of date? |
| old | 365+ days | would merging into a stronger page or pruning beat a refresh? |

**Decay / refresh insight (observed).** Decline *rate* falls with age (66.9% at 31-90 days to 42.6% at 365+), which is the opposite of the usual "older means stale" assumption. Within the same age tier, pages updated in the last 30 days show a lower decline rate than pages last updated 91 to 180 days ago. That is an association. Editors may have chosen which pages to update, and 68% of all pages sit in the 0-30 day tier, so it does not show that refreshing causes recovery.


In [ ]:
# 1) Reason code, action, life stage (the rule)
df["reason_code"] = np.select(
    [down & (df["lost_impressions"] >= 500), down & (df["lost_impressions"] >= 100), down],
    ["decline_large_loss", "decline_moderate_loss", "decline_small_loss"], default="not_declining")
action_map = {"decline_large_loss": "review_for_refresh", "decline_moderate_loss": "watch",
              "decline_small_loss": "no_action", "not_declining": "no_action"}
df["action"] = df["reason_code"].map(action_map)
df["life_stage"] = np.select([df["age_tier_order"] <= 4, df["age_tier_order"] == 5], ["young", "mid"], default="old")
check_first = {"young": "search intent match and indexing", "mid": "outdated facts, dates, links",
               "old": "merge or prune vs refresh"}
df["check_first"] = np.where(df["action"] == "review_for_refresh", df["life_stage"].map(check_first), "")
df["model_support"] = np.where(df["p_tree"] >= 0.6, "supports", "weak")

# 2) Ranked queue: action first, then impressions lost, then tree probability
order = {"review_for_refresh": 0, "watch": 1, "no_action": 2}
df["_o"] = df["action"].map(order)
queue = df.sort_values(["_o", "lost_impressions", "p_tree"], ascending=[True, False, False]).drop(columns="_o").reset_index(drop=True)
queue["priority_rank"] = np.arange(1, len(queue) + 1)
df = df.drop(columns="_o")

print(queue["reason_code"].value_counts().to_string())
print("\nShare of review_for_refresh pages with model_support = supports: %.2f"
      % (queue.loc[queue.action == "review_for_refresh", "model_support"].eq("supports").mean()))
show = ["priority_rank", "content_id", "client_id", "life_stage", "impressions_prev_30d", "impressions_last_30d",
        "lost_impressions", "reason_code", "action", "model_support", "check_first"]
queue.head(20)[show]


reason_code
not_declining            13738
decline_small_loss        6804
decline_moderate_loss     4802
decline_large_loss        4656



Share of review_for_refresh pages with model_support = supports: 0.75


,priority_rank,content_id,client_id,life_stage,impressions_prev_30d,impressions_last_30d,lost_impressions,reason_code,action,model_support,check_first
0,1,content_5fe46e04994d,client_4e07408562,old,218786,120791,97995,decline_large_loss,review_for_refresh,weak,merge or prune vs refresh
1,2,content_8c19996aa890,client_4e07408562,old,161284,89463,71821,decline_large_loss,review_for_refresh,weak,merge or prune vs refresh
2,3,content_9532f197bbc8,client_4e07408562,old,174235,109317,64918,decline_large_loss,review_for_refresh,weak,merge or prune vs refresh
3,4,content_66b4046cc144,client_7f2253d7e2,mid,69303,7408,61895,decline_large_loss,review_for_refresh,supports,"outdated facts, dates, links"
4,5,content_2c2606c5d176,client_19581e27de,mid,164079,104248,59831,decline_large_loss,review_for_refresh,supports,"outdated facts, dates, links"
5,6,content_ec66c58d9826,client_7f2253d7e2,mid,57814,2779,55035,decline_large_loss,review_for_refresh,supports,"outdated facts, dates, links"
6,7,content_cb112fce36be,client_19581e27de,young,124500,72468,52032,decline_large_loss,review_for_refresh,supports,search intent match and indexing
7,8,content_c8e9d6ab9013,client_19581e27de,mid,111885,63326,48559,decline_large_loss,review_for_refresh,supports,"outdated facts, dates, links"
8,9,content_73e5410650b9,client_19581e27de,young,50269,8099,42170,decline_large_loss,review_for_refresh,supports,search intent match and indexing
9,10,content_8d3971bfd976,client_f369cb89fc,young,54506,12652,41854,decline_large_loss,review_for_refresh,weak,search intent match and indexing


In [ ]:
# Decay / refresh insight: decline rate and where the lost impressions sit
decl = df[down]
tier = df.groupby("age_tier").agg(pages=("age_tier", "size")).join(
    decl.groupby("age_tier").agg(declining=("age_tier", "size"), lost=("lost_impressions", "sum")))
tier["decline_rate"] = tier["declining"] / tier["pages"]
tier["share_of_declining_pages"] = tier["declining"] / tier["declining"].sum()
tier["share_of_lost_impressions"] = tier["lost"] / tier["lost"].sum()
print(tier.loc[["31-90", "91-180", "181-365", "365+"], ["pages", "decline_rate", "share_of_declining_pages", "share_of_lost_impressions"]].round(3).to_string())

fresh = df.pivot_table(index="age_tier", columns="freshness_tier", values="trend_direction",
                       aggfunc=[lambda s: (s == "down").mean(), "size"])
fresh.columns = [f"{'decline_rate' if a == '<lambda>' else 'n'}_{b}" for a, b in fresh.columns]
print("\nDecline rate (and n) by age tier x freshness tier:")
print(fresh.loc[["31-90", "91-180", "181-365", "365+"]].round(3).to_string())
print("\nShare of all pages in the 0-30 day freshness tier: %.1f%%" % ((df["freshness_tier"] == "0-30").mean() * 100))


          pages  decline_rate  share_of_declining_pages  share_of_lost_impressions
age_tier                                                                          
31-90       492         0.669                     0.020                      0.017
91-180    11780         0.626                     0.453                      0.448
181-365   11368         0.515                     0.360                      0.381
365+       6360         0.426                     0.167                      0.154

Decline rate (and n) by age tier x freshness tier:
          decline_rate_0-30  decline_rate_181+  decline_rate_31-90  decline_rate_91-180  n_0-30  n_181+  n_31-90  n_91-180
age_tier                                                                                                                  
31-90                 0.669                NaN                 NaN                  NaN   492.0     NaN      NaN       NaN
91-180                0.608                NaN               0.556               

## 2. Intended use and limits

*Who uses this, for what, and where it stops being valid.*

**Intended use.** A content editor or SEO lead uses the weekly list as **decision-support**: pages with a measured recent loss that are worth a human look first. Nothing here changes a page by itself.

**Cost and value (assumptions, editable).** I assume 2 editor-hours per reviewed page and 100 editor-hours a week, so a list of 50 pages. These are placeholders for FlyRank to replace. Value is the *impressions at stake*, meaning what the page lost against the previous 30 days. It is **not** what a refresh would win back, because this data cannot measure that. The weekly list takes at most 10 pages per client, so one client cannot fill it.

**Limits (measured in this notebook and in ML-09).**
- One snapshot of 30,000 pages from 32 clients. No page is followed over time, and nothing shows that refreshing a flagged page helps.
- The tree's precision (about 0.65) sits roughly 10 points above the 0.54 base rate, so `model_support` is a weak tie-breaker, not a verdict.
- The rule flags thousands of pages for review, far more than an editor can read, so the capacity cut-off decides what is read.
- A handful of clients account for most of the lost impressions (Section 3), so one site-wide event can look like many page problems.
- Valid only for pages at least 31 days old with at least 100 impressions in the previous 30 days. New clients and young pages are outside what I measured.


In [ ]:
# Cost / value with explicit assumptions, plus a per-client cap on the weekly list
EDITOR_HOURS_PER_PAGE = 2          # assumption, replace with FlyRank's number
EDITOR_HOURS_PER_WEEK = 100        # assumption
K = EDITOR_HOURS_PER_WEEK // EDITOR_HOURS_PER_PAGE
CAP_PER_CLIENT = 10

cand = queue[queue["action"] == "review_for_refresh"].copy()
cand["client_rank"] = cand.groupby("client_id").cumcount() + 1
weekly = cand[cand["client_rank"] <= CAP_PER_CLIENT].head(K)
queue["weekly_list"] = queue["content_id"].isin(weekly["content_id"])

total_lost = df.loc[down, "lost_impressions"].sum()
uncapped = cand.head(K)
print(f"Weekly capacity: {K} pages ({EDITOR_HOURS_PER_PAGE} h/page x {EDITOR_HOURS_PER_WEEK} h/week, assumptions)")
print(f"review_for_refresh pages: {len(cand):,}  ->  weeks to read them all at {K}/week: {len(cand)/K:.0f}")
print(f"Top {K} uncapped: {uncapped['lost_impressions'].sum()/total_lost:.1%} of all lost impressions, "
      f"from {uncapped['client_id'].nunique()} clients (largest client {uncapped['client_id'].value_counts(normalize=True).iloc[0]:.0%} of the list)")
print(f"Top {K} with cap {CAP_PER_CLIENT}/client: {weekly['lost_impressions'].sum()/total_lost:.1%} of all lost impressions, "
      f"from {weekly['client_id'].nunique()} clients (largest client {weekly['client_id'].value_counts(normalize=True).iloc[0]:.0%} of the list)")
print(f"Impressions at stake per editor-hour (weekly list): {weekly['lost_impressions'].sum()/(K*EDITOR_HOURS_PER_PAGE):,.0f}  [at stake, not recoverable]")
print(f"Tree-supported pages on the weekly list: {weekly['model_support'].eq('supports').mean():.0%}")


Weekly capacity: 50 pages (2 h/page x 100 h/week, assumptions)
review_for_refresh pages: 4,656  ->  weeks to read them all at 50/week: 93
Top 50 uncapped: 13.1% of all lost impressions, from 7 clients (largest client 26% of the list)
Top 50 with cap 10/client: 13.0% of all lost impressions, from 8 clients (largest client 20% of the list)
Impressions at stake per editor-hour (weekly list): 18,352  [at stake, not recoverable]
Tree-supported pages on the weekly list: 62%


**What the numbers show.** The rule marks 4,656 pages `review_for_refresh`; at 50 pages a week that is about 93 weeks of reading, so the cut-offs are too loose to act on as written and the capacity limit does the real selecting. The top 50 pages hold about 13% of all lost impressions, with or without the per-client cap, and the cap lowers the largest client's share of the list from 26% to 20%. The impressions-at-stake figure per editor-hour describes size, not payoff.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**A person must check, for every page on the weekly list:**
1. **Client-level cause first.** If most of that client's pages are declining, treat it as a possible site-wide event (migration, tracking gap, redesign) before touching any page.
2. **Seasonality.** Did this page dip at the same time last year?
3. **Work in progress.** Is someone already updating it, or did a newer page take its traffic?
4. **Data quality.** Are the Search Console numbers for this client complete for both 30-day windows?
5. **Fit of the action.** Use the life-stage "check first" hint, then decide refresh, merge, prune or leave.

The reviewer records one of: act, wait, or reject (with a reason). The reasons feed the monitoring triggers in Section 4.

**What should NOT be automated:**
- Publishing, rewriting or deleting any page.
- Telling a client that a refresh "will recover" traffic. This data cannot show that.
- Acting on a client where at least 75% of pages are declining (my cut-off), until a person checks the client-level cause.
- Using `model_support` alone to include or exclude a page.
- Sharing client names, URLs or page text outside the team (the data is pseudonymised for a reason).


In [ ]:
# Client-level check: which clients have most of their pages declining, and how concentrated is the loss?
cl = df.groupby("client_id").agg(pages=("client_id", "size"), declining_share=("trend_direction", lambda s: (s == "down").mean()),
                                  lost=("lost_impressions", "sum"))
cl["share_of_all_lost"] = cl["lost"] / df["lost_impressions"].sum()
CLIENT_WIDE = 0.75
wide = cl[cl["declining_share"] >= CLIENT_WIDE]
print(f"Clients where at least {CLIENT_WIDE:.0%} of pages are declining: {len(wide)} of {len(cl)}")
print(wide.sort_values("declining_share", ascending=False)[["pages", "declining_share"]].round(3).to_string())

dec_lost = df[down].groupby("client_id")["lost_impressions"].sum().sort_values(ascending=False)
print("\nShare of all lost impressions (declining pages) held by the top 5 clients: %.1f%%" % (dec_lost.head(5).sum() / dec_lost.sum() * 100))
queue["client_wide_decline"] = queue["client_id"].isin(wide.index)
print("Weekly-list pages that sit in a client-wide-decline client:",
      int(queue.loc[queue["weekly_list"], "client_wide_decline"].sum()), "of", int(queue["weekly_list"].sum()))


Clients where at least 75% of pages are declining: 4 of 32
                   pages  declining_share
client_id                                
client_7f2253d7e2   1043            0.937
client_3fdba35f04   2267            0.839
client_9400f1b21c    158            0.766
client_bdd2d3af3a     44            0.750

Share of all lost impressions (declining pages) held by the top 5 clients: 83.5%
Weekly-list pages that sit in a client-wide-decline client: 11 of 50


**What the numbers show.** 4 of 32 clients have at least 75% of their pages declining, and 11 of the 50 weekly-list pages belong to them, so the client-level check is a real first step and not a formality. The top 5 clients hold about 84% of the lost impressions on declining pages, so the list says more about a few sites than about pages in general.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

Light and manual, to be re-run on each new snapshot. The baseline column is this run; the trigger is a rule of thumb I chose, not a tuned value.

| Signal | Baseline (this run) | Trigger | What to do |
|---|---|---|---|
| Share of pages declining | see table below | moves more than 10 points | re-check the 100 and 500 cut-offs |
| Median `impressions_prev_30d` | see table below | moves more than 30% | rebuild the cut-offs |
| Tree precision on newly measured pages | see table below | falls below base rate + 5 points | drop `model_support` from the list |
| Largest client's share of lost impressions | see table below | above 30% | review that client before page by page |
| Reviewer rejects flagged pages | not measured yet | over half rejected for non-page reasons | revise the rule |
| Calendar | this snapshot | monthly at least | re-run and compare |

**Retrain only when** a newer snapshot with measured outcomes exists, so the model can be scored on pages it has not seen. Nothing in this snapshot can test a forward prediction.


In [ ]:
baseline = pd.DataFrame([
    ["Share of pages declining", round(float(base_rate), 3), "moves more than 10 points"],
    ["Median impressions_prev_30d", float(df["impressions_prev_30d"].median()), "moves more than 30%"],
    ["Tree precision (grouped, out-of-fold)", round(float(precision_score(y, tree_pred)), 3), "below base rate + 0.05"],
    ["Largest client share of lost impressions (declining pages)", round(float(dec_lost.max() / dec_lost.sum()), 3), "above 0.30"],
], columns=["signal", "baseline_this_run", "trigger"])
baseline


,signal,baseline_this_run,trigger
0,Share of pages declining,0.542,moves more than 10 points
1,Median impressions_prev_30d,210.000,moves more than 30%
2,"Tree precision (grouped, out-of-fold)",0.645,below base rate + 0.05
3,Largest client share of lost impressions (decl...,0.276,above 0.30


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

The notebook writes three files:
- `work/outputs/playbook_action_queue.csv`: the full ranked queue (stays out of git by design; this notebook regenerates it).
- `work/outputs/playbook_metrics.json`: the numbers the paper can trace back to (commit this).
- `work/figures/playbook_loss_concentration.png`: one figure for the paper (commit this).

In Colab these files are written to the notebook's temporary workspace, not to GitHub. Download the figure and the JSON and add them to the repo.


In [ ]:
os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)

out_cols = ["priority_rank", "content_id", "client_id", "age_tier", "life_stage", "freshness_tier",
            "impressions_prev_30d", "impressions_last_30d", "lost_impressions", "reason_code", "action",
            "check_first", "model_support", "p_tree", "weekly_list", "client_wide_decline"]
queue[out_cols].to_csv("work/outputs/playbook_action_queue.csv", index=False)

metrics = {
    "pages": int(len(df)), "clients": int(df["client_id"].nunique()),
    "share_declining": round(float(base_rate), 4),
    "action_counts": queue["action"].value_counts().to_dict(),
    "reason_counts": queue["reason_code"].value_counts().to_dict(),
    "weekly_capacity": int(K), "cap_per_client": int(CAP_PER_CLIENT),
    "weekly_list_share_of_lost_impressions": round(float(weekly["lost_impressions"].sum() / total_lost), 4),
    "top5_clients_share_of_lost_impressions": round(float(dec_lost.head(5).sum() / dec_lost.sum()), 4),
    "tree_precision_grouped_oof": round(float(precision_score(y, tree_pred)), 4),
    "tree_recall_grouped_oof": round(float(recall_score(y, tree_pred)), 4),
    "assumptions": {"editor_hours_per_page": EDITOR_HOURS_PER_PAGE, "editor_hours_per_week": EDITOR_HOURS_PER_WEEK},
}
json.dump(metrics, open("work/outputs/playbook_metrics.json", "w"), indent=2)

# One figure: how concentrated are the lost impressions across declining pages?
lost_sorted = np.sort(df.loc[down, "lost_impressions"].values)[::-1]
cum = np.cumsum(lost_sorted) / lost_sorted.sum()
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(np.arange(1, len(cum) + 1), cum * 100, color="#1f4e79", lw=2)
for k in (50, 500):
    ax.scatter([k], [cum[k - 1] * 100], color="#c0392b", zorder=3)
    ax.annotate(f"top {k}: {cum[k-1]*100:.0f}%", (k, cum[k - 1] * 100), textcoords="offset points", xytext=(8, -14), fontsize=9)
ax.set_xscale("log"); ax.set_xlabel("Declining pages, ranked by impressions lost (log scale)")
ax.set_ylabel("Share of all lost impressions (%)")
ax.set_title(f"500 of {len(cum):,} declining pages hold {cum[499]*100:.0f}% of the lost impressions", fontsize=11, loc="left")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout(); fig.savefig("work/figures/playbook_loss_concentration.png", dpi=150)
print(sorted(os.listdir("work/outputs")), sorted(os.listdir("work/figures")))
print(json.dumps(metrics, indent=2)[:700])


['playbook_action_queue.csv', 'playbook_metrics.json'] ['playbook_loss_concentration.png']
{
  "pages": 30000,
  "clients": 32,
  "share_declining": 0.5421,
  "action_counts": {
    "no_action": 20542,
    "watch": 4802,
    "review_for_refresh": 4656
  },
  "reason_counts": {
    "not_declining": 13738,
    "decline_small_loss": 6804,
    "decline_moderate_loss": 4802,
    "decline_large_loss": 4656
  },
  "weekly_capacity": 50,
  "cap_per_client": 10,
  "weekly_list_share_of_lost_impressions": 0.1302,
  "top5_clients_share_of_lost_impressions": 0.8354,
  "tree_precision_grouped_oof": 0.6447,
  "tree_recall_grouped_oof": 0.8615,
  "assumptions": {
    "editor_hours_per_page": 2,
    "editor_hours_per_week": 100
  }
}


## Demo outline (5 minutes)

**0:00–0:45 · The question.** A content team has more pages losing search traffic than hours to fix them. Which pages should a person look at first this week? Old content is often assumed to be stale, so I tested that assumption before building anything.

**0:45–1:45 · The method.** 30,000 pseudonymised pages from 32 clients. A page is "declining" if its impressions in the last 30 days fell more than 20% below the 30 days before. I compared three options: flag every page, a one-line age rule, and a depth-4 decision tree. I validated with folds grouped by client, so every score comes from clients the model had not seen. I also audited leakage: the 90-day impressions column contains the label window, so I rebuilt features from windows that come before it.

**1:45–2:45 · The chart.** Share of pages declining by age tier (decline_by_age.png). It falls from 66.9% for the youngest pages to 42.6% for pages a year old or more. This is the opposite of "older means stale".

**2:45–3:45 · One honest result.** 54.2% of pages are declining. The tree's precision is 0.64 against a 0.54 base rate, and the age rule's is 0.63. So the tree barely beats a one-line rule. Its higher recall comes mostly from flagging more pages, not from being smarter.

**3:45–4:30 · One recommendation.** Use the model as a ranked, human-reviewed queue, not a verdict. Put the pages that lost the most impressions first, cap the queue at about 50 pages a week with at most 10 per client, and re-check it monthly. Loss is concentrated: the top 5 clients hold about 84% of lost impressions.

**4:30–5:00 · The limit.** This is one snapshot. Nothing here shows that refreshing a flagged page brings its traffic back. The honest next step is a small test: refresh some flagged pages and leave similar ones alone, then compare.

In [ ]:
## Shareable cuts

**Social post (methodology)**

Before building a model to decide which web pages to refresh first, I tested the common belief that "older content is stale." In 30,000 pseudonymised pages, the opposite was true: 67% of the newest pages were losing search impressions, versus 43% of year-old ones. Then I found a leak in my own features. One column contained the very window I was trying to predict, and it made the model look far better than it was. After fixing it and validating on clients the model had never seen, a decision tree (64% precision) barely beat a one-line age rule (63%). Lesson: check your assumptions and your timeline before you trust a score. #MachineLearning #DataScience #SEO

**Employer-facing summary (3 sentences)**

I built a decision-support tool that ranks web pages for content refresh, using a leakage-audited decision tree on 30,000 pseudonymised pages from 32 clients in FlyRank's internship dataset. It showed that newer pages were declining more than older ones, and that the model beat a simple age rule by only about one point of precision, so I recommended a human-reviewed weekly queue of about 50 pages rather than an automated verdict. The work is published as a research paper with the code, validation and limitations in a public GitHub repo.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.